In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


#  کانفیگ

CONFIGS = {
    "song": {
        "path": "spotify_songs.csv",
        "title": "track_name",
        "author": "track_artist",
        "category": "playlist_genre",
    },
    "movie": {
        "path": "movie.csv",   
        "title": "Title",
        "author": "Description",
        "category": "Genre",
    },
}


class Recommender:
    def __init__(self, cfg):
        if not os.path.exists(cfg["path"]):
            raise FileNotFoundError(f"f فایل {cfg['path']}")

        self.cfg = cfg
        self.df = pd.read_csv(cfg["path"])
        print(f" [ ] {len(self.df)}  لود شد از {cfg['path']}")

        # پاکسازی
        for col in [cfg["title"], cfg["author"], cfg["category"]]:
            self.df[col] = self.df[col].fillna("").astype(str)

        #(out put) متن ترکیبی
        text = (self.df[cfg["title"]] + " " +
                self.df[cfg["author"]] + " " +
                self.df[cfg["category"]])

        # TF-IDF
        self.vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
        self.matrix = self.vec.fit_transform(text)

    def recommend(self, title, author="", category="", top_n=5):
        cfg = self.cfg
        query = f"{title} {author} {category}".strip()
        q_vec = self.vec.transform([query])

        # شباهت
        scores = (self.matrix @ q_vec.T).toarray().flatten()
        same = self.df[cfg["title"]].str.lower() == title.lower()
        scores[same.values] = -np.inf

        top = np.argsort(scores)[::-1][:top_n]

        result = self.df.iloc[top][[cfg["title"], cfg["author"], cfg["category"]]].copy()
        result["score"] = scores[top]
        return result.reset_index(drop=True)



#  (run) اجرا

def run(kind):
    cfg = CONFIGS[kind]
    engine = Recommender(cfg)

    print(f"\n حالت <=> {'آهنگ' if kind == 'song' else 'فیلم'}")
    title    = input("اسم فیلم یا آهنگی که دوست داری چیه ؟ (بیشتر به چه آهنگ/فیلم علاقه داری ؟)").strip()
    author   = input("خواننده/کارگردان مورد علاقه‌ت کیه؟ (مانند : Taylor Swift و Christopher Nolan)").strip()
    category = input("ژانر مورد علاقه‌ت چیه؟/(Action/Drama/Comedy | rap/pop/rock)").strip()

    results = engine.recommend(title, author, category, top_n=5)

    print("\n<= پیشنهادها:")
    for i, row in enumerate(results.itertuples(index=False), 1):
        vals = list(row)
        score = vals[-1]
        name = vals[0]
        info = " | ".join(str(v)[:80] for v in vals[1:-1] if v)
        print(f"{i}. {name}  =>  {info}  (امتیاز: {score:.3f})")


if __name__ == "__main__":
    mode = input("فیلم میخوای یا آهنگ؟ (movie/song) ").strip().lower() or "song"
    run(mode)